In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from capstone.config import CATEGORICAL, FEATURES, NUMERIC, PLOT, STYLES, FrankensteinConfig, LossWeights
from capstone.data import load_recipes, make_frankensteins
from capstone.features import transform
from capstone.models import adjust_many, affinity, build_search_space, distances, fit_centroids, predict
from capstone.utilities import setup_logging

setup_logging("WARNING")

pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 50)
plt.rcParams["figure.dpi"] = 110
COLORS = PLOT.style_colors
SEED = 0

In [ ]:
df = load_recipes()
model = fit_centroids(df[FEATURES], df["style"])
space = build_search_space(model, df)

N_FRANK = 300
frank = make_frankensteins(df, FrankensteinConfig(n=N_FRANK, seed=SEED))
frank[FEATURES + ["n_donor_styles"]].head()

In [ ]:
# What do the frankensteins look like to the model before any adjustment?
frank["predicted"] = predict(model, frank)
print(frank["predicted"].value_counts())
print(frank["n_donor_styles"].value_counts().sort_index())
fd = distances(model, frank)
fd.describe().T

In [ ]:
# Frankenstein feature distributions vs. the real recipes they were sampled from
fig, axes = plt.subplots(1, len(NUMERIC), figsize=(16, 3))
for ax, f in zip(axes, NUMERIC):
    bins = np.histogram_bin_edges(pd.concat([df[f], frank[f]]), bins=12)
    ax.hist(df[f], bins=bins, alpha=0.5, density=True, label="real")
    ax.hist(frank[f], bins=bins, alpha=0.5, density=True, label="frankenstein")
    ax.set_title(f)
axes[0].legend()
fig.tight_layout()

In [ ]:
# Adjust every frankenstein toward each style centroid
runs = {}
for target in STYLES:
    adjusted, info = adjust_many(frank, target, space)
    runs[target] = {"adjusted": adjusted, "info": info}

def summarize(frank, runs, model):
    rows = []
    for target, r in runs.items():
        before = distances(model, frank)[target]
        after = distances(model, r["adjusted"])[target]
        aff_before = affinity(model, frank)[target]
        aff_after = affinity(model, r["adjusted"])[target]
        pred = predict(model, r["adjusted"])
        info = r["info"]
        rows.append({
            "target": target,
            "success (nearest = target)": (pred == target).mean(),
            "already target": (frank["predicted"] == target).mean(),
            "dist before": before.mean(), "dist after": after.mean(),
            "affinity before": aff_before.mean(), "affinity after": aff_after.mean(),
            "mean type changes": info["n_type_changes"].mean(),
            "mean added": info["n_added"].mean(), "mean removed": info["n_removed"].mean(),
            "frac any add": (info["n_added"] > 0).mean(),
            "frac any type change": (info["n_type_changes"] > 0).mean(),
            "mean total loss": info["loss_total"].mean(),
        })
    return pd.DataFrame(rows).set_index("target")

summarize(frank, runs, model).T

In [ ]:
# Loss decomposition per target
pd.concat({t: r["info"].filter(like="loss_").mean() for t, r in runs.items()}, axis=1)

In [ ]:
# Ratio changes in original units (only for ingredients present before and after)
rows = []
for target, r in runs.items():
    delta = (r["adjusted"][NUMERIC] - frank[NUMERIC]).abs()
    kept = (frank[NUMERIC] > 0) & (r["adjusted"][NUMERIC] > 0)
    d = delta.where(kept)
    rows.append(d.mean().rename(target))
pd.concat(rows, axis=1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, target in zip(axes, STYLES):
    delta = (runs[target]["adjusted"][NUMERIC] - frank[NUMERIC]).where((frank[NUMERIC] > 0) & (runs[target]["adjusted"][NUMERIC] > 0))
    ax.boxplot([delta[f].dropna() for f in NUMERIC], tick_labels=NUMERIC)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_title(f"toward {target}")
axes[0].set_ylabel("change in baker's %")
fig.tight_layout()

In [ ]:
# Which type swaps and add/remove operations does the optimizer choose?
rows = []
for target, r in runs.items():
    adj = r["adjusted"]
    for col in CATEGORICAL:
        changed = frank[col] != adj[col]
        for (a, b), n in pd.DataFrame({"from": frank.loc[changed, col], "to": adj.loc[changed, col]}).value_counts().items():
            rows.append({"target": target, "feature": col, "from": a, "to": b, "count": n})
swaps = pd.DataFrame(rows)
swaps.sort_values(["target", "count"], ascending=[True, False]).groupby("target").head(8)

In [ ]:
# Before/after distance to the target centroid
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharex=True, sharey=True)
for ax, target in zip(axes, STYLES):
    before = distances(model, frank)[target]
    after = distances(model, runs[target]["adjusted"])[target]
    ax.scatter(before, after, s=12, alpha=0.6, color=COLORS[target])
    lim = [0, before.max() * 1.05]
    ax.plot(lim, lim, "k--", lw=0.8)
    ax.set_title(f"toward {target}")
    ax.set_xlabel("distance before")
axes[0].set_ylabel("distance after")
fig.tight_layout()

In [ ]:
# Example before / after
def show(recipe_id, target):
    before = frank.loc[recipe_id, FEATURES]
    after = runs[target]["adjusted"].loc[recipe_id]
    out = pd.DataFrame({"before": before, "after": after})
    out["changed"] = out["before"] != out["after"]
    return out

print(frank.loc["frank_0000", ["predicted"]].iloc[0], "->", "neapolitan")
display(show("frank_0000", "neapolitan"))
display(runs["neapolitan"]["info"].loc["frank_0000"])

In [ ]:
# Does the adjusted recipe look like a real recipe of the target style? Distance to nearest real target recipe
from sklearn.metrics import pairwise_distances

Zreal = transform(model, df)
real_target = {s: Zreal[(df["style"] == s).values] for s in STYLES}
rows = []
for target, r in runs.items():
    Zb, Za = transform(model, frank), transform(model, r["adjusted"])
    rows.append({
        "target": target,
        "nearest real (before)": pairwise_distances(Zb, real_target[target]).min(axis=1).mean(),
        "nearest real (after)": pairwise_distances(Za, real_target[target]).min(axis=1).mean(),
    })
pd.DataFrame(rows).set_index("target")

In [ ]:
# Bootstrap over frankensteins: uncertainty on success rate and mean distance reduction
rng = np.random.default_rng(SEED)
B = 1000
boot = []
for target in STYLES:
    ok = (predict(model, runs[target]["adjusted"]) == target).to_numpy()
    gain = (distances(model, frank)[target] - distances(model, runs[target]["adjusted"])[target]).to_numpy()
    n_changes = (runs[target]["info"][["n_type_changes", "n_added", "n_removed"]].sum(axis=1)).to_numpy()
    for _ in range(B):
        idx = rng.integers(0, len(ok), len(ok))
        boot.append({"target": target, "success": ok[idx].mean(), "dist_gain": gain[idx].mean(), "discrete_changes": n_changes[idx].mean()})
boot = pd.DataFrame(boot)
boot.groupby("target").agg(["mean", lambda s: s.quantile(0.025), lambda s: s.quantile(0.975)]).set_axis(
    pd.MultiIndex.from_product([["success", "dist_gain", "discrete_changes"], ["mean", "ci_lo", "ci_hi"]]), axis=1
)

In [ ]:
# Stability across independent frankenstein draws (different seeds), not just resampling one draw
rows = []
for seed in range(1, 21):
    fr = make_frankensteins(df, FrankensteinConfig(n=100, seed=seed))
    for target in STYLES:
        adj, info = adjust_many(fr, target, space)
        rows.append({
            "seed": seed, "target": target,
            "success": (predict(model, adj) == target).mean(),
            "dist_gain": (distances(model, fr)[target] - distances(model, adj)[target]).mean(),
            "frac_add": (info["n_added"] > 0).mean(),
        })
seeds = pd.DataFrame(rows)
seeds.groupby("target")[["success", "dist_gain", "frac_add"]].agg(["mean", "std", "min", "max"])

In [ ]:
# Sensitivity to the loss weights: sweep one at a time on a fixed frankenstein set
sub = frank.iloc[:150]

def run_weights(weights):
    rows = []
    for target in STYLES:
        a, info = adjust_many(sub, target, space, weights)
        rows.append({
            "target": target,
            "success": (predict(model, a) == target).mean(),
            "dist_after": distances(model, a)[target].mean(),
            "ratio_change_z": info["loss_ratio"].mean(),
            "type_changes": info["n_type_changes"].mean(),
            "added": info["n_added"].mean(),
            "removed": info["n_removed"].mean(),
        })
    return pd.DataFrame(rows).drop(columns="target").mean()

sweeps = {
    "add": [0.0, 0.5, 1.0, 2.0, 5.0, 10.0, 50.0],
    "type_change": [0.0, 0.25, 0.5, 1.0, 2.0, 5.0, 20.0],
    "ratio": [0.0, 0.01, 0.1, 0.5, 1.0, 5.0, 50.0],
}
sweep_results = {}
for name, values in sweeps.items():
    sweep_results[name] = pd.DataFrame({v: run_weights(LossWeights(**{name: v})) for v in values}).T
    sweep_results[name].index.name = name
    display(sweep_results[name])

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for col, (name, res) in enumerate(sweep_results.items()):
    x = np.arange(len(res))
    axes[0, col].plot(x, res["success"], marker="o", label="success")
    axes[0, col].plot(x, 1 - res["dist_after"] / res["dist_after"].max(), marker="s", ls="--", label="1 - dist (scaled)")
    axes[0, col].set_xticks(x, res.index)
    axes[0, col].set_title(f"{name} weight")
    axes[0, col].legend()
    for k in ["type_changes", "added", "removed"]:
        axes[1, col].plot(x, res[k], marker="o", label=k)
    axes[1, col].plot(x, res["ratio_change_z"], marker="s", ls="--", label="ratio change (z^2)")
    axes[1, col].set_xticks(x, res.index)
    axes[1, col].set_xlabel(f"{name} weight")
    axes[1, col].legend()
fig.tight_layout()

In [ ]:
# Real recipes as a sanity check: adjusting a real recipe toward its own style should do almost nothing,
# toward another style should move it
rows = []
for target in STYLES:
    a, info = adjust_many(df, target, space)
    own = (df["style"] == target).to_numpy()
    for label, mask in [("own style", own), ("other style", ~own)]:
        rows.append({
            "target": target, "recipes": label, "n": mask.sum(),
            "success": (predict(model, a[mask]) == target).mean(),
            "type_changes": info.loc[mask, "n_type_changes"].mean(),
            "added": info.loc[mask, "n_added"].mean(),
            "removed": info.loc[mask, "n_removed"].mean(),
            "loss_ratio": info.loc[mask, "loss_ratio"].mean(),
        })
pd.DataFrame(rows).set_index(["target", "recipes"])